# 2023 — Quantization: Running Big Models in Low Precision

**Papers:** Dettmers et al., *"LLM.int8()"* (2022), the outlier features; Elias Frantar et al., *"GPTQ: Accurate Post-Training Quantization for Generative Pre-trained Transformers"* (2022/ICLR 2023); Lin et al., *AWQ* (2023); Dettmers et al., *QLoRA* (2023), which introduced 4-bit NF4 and finetuning on top. In 2025–2026, models also ship or train natively in FP8 and FP4 (e.g. DeepSeek-V3/V4, gpt-oss MXFP4).

Weights stored in 16 bits take 2 bytes per parameter. A 70B model needs 140 GB just to load. Storing weights in **8, 4 or even 3 bits** cuts memory and memory bandwidth (the real bottleneck when generating) 2–5×.

Quantization maps each weight to one of $2^b$ levels: $\hat w = s \cdot \operatorname{round}(w / s)$, with scale $s = \max|w| / (2^{b-1}-1)$. Key choices:
- **Granularity:** one scale per tensor, per output channel, or per **group** of 32–128 weights. Finer is more accurate.
- **Outliers:** a few huge values stretch the scale and wipe out precision for everything else.
- **GPTQ:** quantize the columns of a weight matrix one at a time, and **adjust the not-yet-quantized columns to compensate** for each rounding error, using second-order information $H = X^\top X$ from a little calibration data.

In [1]:
import os, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

def get_device():
    """CUDA > Apple MPS > CPU. Force one with the env var NN_DEVICE=cpu|cuda|mps."""
    forced = os.environ.get("NN_DEVICE")
    if forced:
        return torch.device(forced)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def seed_everything(seed=0):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

device = get_device()
torch.set_flush_denormal(True)   # tiny "denormal" floats are extremely slow on CPUs; treat them as 0
seed_everything(0)
print("PyTorch", torch.__version__, "| device:", device)
import copy

PyTorch 2.15.0 | device: mps


In [2]:
# A procedurally generated corpus (~290K characters) with a real held-out split.
_rng = random.Random(0)
_subjects = ["the model", "a small network", "the optimizer", "each layer", "the attention head", "the router", "a student", "the teacher"]
_verbs = ["learns", "predicts", "updates", "compresses", "routes", "remembers", "forgets", "copies"]
_objects = ["the next token", "a hidden state", "the gradient", "its weights", "a long context", "the key and value", "every expert", "the loss"]
_advs = ["quickly", "slowly", "carefully", "in parallel", "step by step", "without labels", "at scale", "once again"]
def _sentence():
    s = f"{_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)} {_rng.choice(_advs)}"
    if _rng.random() < 0.4: s += f" because {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    if _rng.random() < 0.2: s += f", then {_rng.choice(_subjects)} {_rng.choice(_verbs)} {_rng.choice(_objects)}"
    return s[0].upper() + s[1:] + ". "
text = "".join(_sentence() for _ in range(4000))
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}
data = torch.tensor([stoi[c] for c in text], device=device)
split = int(0.9 * len(data)); train_data, val_data = data[:split], data[split:]
encode = lambda s: [stoi[c] for c in s]; decode = lambda ids: "".join(chars[i] for i in ids)
print(f"{len(text):,} characters, vocab {V}")

290,321 characters, vocab 31


In [3]:
class Block(nn.Module):
    def __init__(self, d, heads):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)
        self.qkv, self.proj, self.h = nn.Linear(d, 3 * d), nn.Linear(d, d), heads
        self.mlp = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))
    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.ln1(x)).view(B, T, 3, self.h, D // self.h).permute(2, 0, 3, 1, 4)
        x = x + self.proj(F.scaled_dot_product_attention(q, k, v, is_causal=True).transpose(1, 2).reshape(B, T, D))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    """A small GPT-2-style character model (pre-norm blocks, learned positions)."""
    def __init__(self, V, d=128, n_layers=4, heads=4, ctx=128):
        super().__init__()
        self.ctx = ctx
        self.tok, self.pos = nn.Embedding(V, d), nn.Embedding(ctx, d)
        self.blocks = nn.ModuleList([Block(d, heads) for _ in range(n_layers)])
        self.ln, self.head = nn.LayerNorm(d), nn.Linear(d, V, bias=False)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        for b in self.blocks: x = b(x)
        return self.head(self.ln(x))

def train_gpt(model, train_data, steps=800, B=32, lr=2e-3):
    model = model.to(train_data.device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.1)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, lr, total_steps=steps, pct_start=0.1)
    for s in range(steps):
        ix = torch.randint(0, len(train_data) - model.ctx - 1, (B,), device=train_data.device)
        x = train_data[ix[:, None] + torch.arange(model.ctx, device=train_data.device)]; y = train_data[ix[:, None] + torch.arange(1, model.ctx + 1, device=train_data.device)]
        loss = F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1))
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
    return model.eval()

@torch.no_grad()
def eval_loss(model, data, n=20, B=32):
    g = torch.Generator(device="cpu").manual_seed(123); losses = []
    for _ in range(n):
        ix = torch.randint(0, len(data) - model.ctx - 1, (B,), generator=g).to(data.device)
        x = data[ix[:, None] + torch.arange(model.ctx, device=data.device)]; y = data[ix[:, None] + torch.arange(1, model.ctx + 1, device=data.device)]
        losses.append(F.cross_entropy(model(x).reshape(-1, model.head.out_features), y.reshape(-1)).item())
    return float(np.mean(losses))

seed_everything(0)
model = train_gpt(GPT(V, d=128, n_layers=4, heads=4), train_data, steps=800)
base_loss = eval_loss(model, val_data)
print(f"full-precision (fp32) model: val loss {base_loss:.4f}")

full-precision (fp32) model: val loss 0.2148


## Round-to-nearest (RTN) at different granularities

In [4]:
def quantize(w, bits, group=None, per_channel=True):
    """Symmetric round-to-nearest. w: (out, in). group: quantize each run of `group` input weights with its own scale."""
    qmax = 2 ** (bits - 1) - 1
    if group:
        wg = w.reshape(w.shape[0], -1, group); s = wg.abs().amax(-1, keepdim=True) / qmax
        return (torch.round(wg / s).clamp(-qmax - 1, qmax) * s).reshape_as(w)
    s = (w.abs().amax(1, keepdim=True) if per_channel else w.abs().max()) / qmax
    return torch.round(w / s).clamp(-qmax - 1, qmax) * s

def quantized_copy(fn):
    m = copy.deepcopy(model)
    for name, mod in m.named_modules():
        if isinstance(mod, nn.Linear) and name != "head":
            mod.weight.data = fn(mod.weight.data)
    return m

rows = []
for bits in [8, 4, 3]:
    for label, kw in [("per-tensor", dict(per_channel=False)), ("per-channel", {}), ("group-32", dict(group=32))]:
        L = eval_loss(quantized_copy(lambda w: quantize(w, bits, **kw)), val_data); rows.append((bits, label, L))
        print(f"int{bits} {label:12s} val loss {L:.4f}  (+{L - base_loss:.4f})")

int8 per-tensor   val loss 0.2148  (+0.0000)


int8 per-channel  val loss 0.2148  (+0.0000)


int8 group-32     val loss 0.2148  (+0.0000)


int4 per-tensor   val loss 0.2199  (+0.0051)


int4 per-channel  val loss 0.2158  (+0.0010)


int4 group-32     val loss 0.2158  (+0.0010)


int3 per-tensor   val loss 0.2930  (+0.0782)


int3 per-channel  val loss 0.2311  (+0.0163)


int3 group-32     val loss 0.2244  (+0.0096)


## The outlier problem
LLMs develop a few **outlier channels** with huge magnitudes (LLM.int8() found them in every model above ~6.7B parameters). We plant one large weight per output row and look at what happens to everything else in the row.

In [5]:
w = model.blocks[0].mlp[0].weight.data.clone()
w_out = w.clone(); w_out[:, 5] *= 40                                                  # one outlier input channel
for name, W in [("normal weights", w), ("with an outlier channel", w_out)]:
    err_tensor = (quantize(W, 4, per_channel=False) - W)[:, 40:].abs().mean().item()        # columns outside the outlier's group
    err_group = (quantize(W, 4, group=32) - W)[:, 40:].abs().mean().item()
    print(f"{name:26s} int4 error on the *other* weights: per-tensor {err_tensor:.4f} | group-32 {err_group:.4f}")
print("Per-tensor scaling lets one outlier channel ruin every weight; with groups of 32, weights outside the outlier's group are unaffected.")

normal weights             int4 error on the *other* weights: per-tensor 0.0073 | group-32 0.0042
with an outlier channel    int4 error on the *other* weights: per-tensor 0.0476 | group-32 0.0042
Per-tensor scaling lets one outlier channel ruin every weight; with groups of 32, weights outside the outlier's group are unaffected.


## GPTQ: error-compensating quantization
For a linear layer $y = Wx$, minimise the output error $\|WX - \hat W X\|^2$ on calibration inputs $X$. With $H = 2XX^\top$, GPTQ goes column by column: quantize column $j$, then spread its error over the remaining columns using row $j$ of the inverse Hessian (computed once via a Cholesky factorisation).

In [6]:
@torch.no_grad()
def gptq(W, H, bits, group=32, damp=0.01):
    W = W.clone().float(); n = W.shape[1]
    H = H + damp * H.diag().mean() * torch.eye(n, device=W.device)
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True)   # upper Cholesky of H^-1
    Q = torch.zeros_like(W); qmax = 2 ** (bits - 1) - 1
    for j in range(n):
        if j % group == 0:                                                            # scale for the next group, from the updated weights
            s = W[:, j:j + group].abs().amax(1, keepdim=True).clamp_min(1e-8) / qmax
        q = torch.round(W[:, j:j + 1] / s).clamp(-qmax - 1, qmax) * s
        Q[:, j:j + 1] = q
        err = (W[:, j:j + 1] - q) / Hinv[j, j]
        W[:, j + 1:] -= err * Hinv[j, j + 1:][None]                                   # compensate the remaining columns
    return Q

def collect_hessians(m, n_batches=8):
    """Calibration: record H = 2 X^T X of the input to every Linear layer."""
    H, hooks = {}, []
    for name, mod in m.named_modules():
        if isinstance(mod, nn.Linear) and name != "head":
            H[name] = torch.zeros(mod.in_features, mod.in_features, device=device)
            def hook(mod, inp, out, name=name):                      # must return None: a returned value would replace the output
                x = inp[0].reshape(-1, inp[0].shape[-1]); H[name].add_(2 * x.T @ x)
            hooks.append(mod.register_forward_hook(hook))
    with torch.no_grad():
        for _ in range(n_batches):
            ix = torch.randint(0, len(train_data) - m.ctx, (8,), device=device)
            m(train_data[ix[:, None] + torch.arange(m.ctx, device=train_data.device)])
    for h in hooks: h.remove()
    return H

Hs = collect_hessians(model)
def gptq_copy(bits):
    m = copy.deepcopy(model)
    for name, mod in m.named_modules():
        if name in Hs: mod.weight.data = gptq(mod.weight.data, Hs[name], bits)
    return m

print(f"{'bits':>5s} {'RTN group-32':>14s} {'GPTQ group-32':>14s}   (fp32 baseline {base_loss:.4f})")
for bits in [4, 3, 2]:
    rtn = eval_loss(quantized_copy(lambda w: quantize(w, bits, group=32)), val_data)
    gq = eval_loss(gptq_copy(bits), val_data)
    print(f"int{bits:<2d} {rtn:14.4f} {gq:14.4f}")

 bits   RTN group-32  GPTQ group-32   (fp32 baseline 0.2148)


int4          0.2158         0.2151


int3          0.2244         0.2173


int2          0.8012         0.4475


## Memory

In [7]:
n = sum(p.numel() for p in model.parameters())
for label, b in [("fp32", 32), ("bf16", 16), ("int8", 8 + 0.5), ("int4 (group-32, 16-bit scales)", 4 + 0.5), ("int3 (group-32)", 3 + 0.5)]:
    print(f"{label:32s} {n * b / 8 / 1e6:6.2f} MB for this model | a 70B model would need {70e9 * b / 8 / 1e9:6.0f} GB")

fp32                               3.27 MB for this model | a 70B model would need    280 GB
bf16                               1.64 MB for this model | a 70B model would need    140 GB
int8                               0.87 MB for this model | a 70B model would need     74 GB
int4 (group-32, 16-bit scales)     0.46 MB for this model | a 70B model would need     39 GB
int3 (group-32)                    0.36 MB for this model | a 70B model would need     31 GB
